# airfoil relation-learner — scale loop on Colab (free GPU, no Kaggle wait)

The box (CPU) validated every organ of the factored proposer (see `incubation/evolve/learner/LEARNER.md`). This runs the **scale loop** on Colab. Phase A (recognizer-guided loop over rich data) is CPU-bound — it runs even on a CPU runtime; switch to a **T4 GPU runtime** only for Phase B (the learned suffix policy).

**Run order:** set `GH_TOKEN` below (a GitHub PAT with read access to the private repo), then Runtime → Run all.

In [ ]:
GH_TOKEN = ""  # <-- paste a GitHub read token for asystemoffields/airfoil (private)
!pip -q install huggingface_hub 2>/dev/null
import os
url = f"https://{GH_TOKEN}@github.com/asystemoffields/airfoil.git" if GH_TOKEN else "https://github.com/asystemoffields/airfoil.git"
!git clone --depth 1 {url} /content/airfoil 2>&1 | tail -2
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# paths + data. The learner imports grammar/harness/dsl from the repo subtree; ConceptARC is fetched from HF.
import sys, json, numpy as np
L = "/content/airfoil/incubation/evolve/learner"
for p in (L, L + "/../", L + "/../../arc"):
    sys.path.insert(0, os.path.abspath(p))
os.chdir(L)
from huggingface_hub import hf_hub_download
import shutil
os.makedirs("/content/conceptarc", exist_ok=True)
for fn in ["arc-agi_concept-challenges.json", "arc-agi_concept-solutions.json"]:
    p = hf_hub_download("imone/conceptarc-converted", fn, repo_type="dataset")
    shutil.copy(p, f"/content/conceptarc/{fn}")
import ground_conceptarc as GC
GC.CH = "/content/conceptarc/arc-agi_concept-challenges.json"
GC.SO = "/content/conceptarc/arc-agi_concept-solutions.json"
print("weights present:", os.path.exists("learner_v2h.pt"), "| ConceptARC tasks:", len(list(GC.load_conceptarc())))

In [ ]:
# PHASE A — scale the recognizer-guided expert-iteration loop over BARC (stream) + ConceptARC (real).
import kaggle_loop as K
from ground_barc import load_barc_tasks  # streams BARC ARC-Heavy as (tid,train,test,meta) over https
N = 3000
K.loop(((tid, tr, te) for tid, tr, te, meta in load_barc_tasks(limit=N)), f"BARC ARC-Heavy (N={N})")
K.loop(((t, tr, te) for t, tr, te in GC.load_conceptarc()), "ConceptARC (real)")
print("\nPHASE A done — real-data coverage + library growth at scale with the CURRENT (recolor/select) effects.")

## Phase B (the GO bet — build here on Colab)
Phase A coverage is bounded by our thin EFFECT vocabulary (recolor/select). **Grow the effect leaves** (move/draw/fill/construct — *earned*, not hand-coded) from verified solves + BARC concept-tags, cold-start the **learned suffix policy** (T4), run **verifier-as-reward expert iteration**, and apply the **leave-one-FAMILY-out GO gate**: GO to a full loop iff held-out-family `beyond_gen6 ≥ 3`. The box result is the floor — do not regress it.